# class-label-sources — probe 4: can an open-vocabulary model label our classes from a prompt?

**Status:** `failed`

## The question

Skip training entirely and just *name* the classes. `ultralytics` ships **YOLOE**
— YOLO11 segmentation with a CLIP text encoder — so `set_classes(names, get_text_pe(names))`
returns masks for concepts it was never explicitly trained to segment.

We need four classes OSM cannot supply well: `building`, `track`, `woody`, `water`.

## What would make this `failed`

Written before running anything, which is the only defence against reading a
number generously afterwards:

> The model must place masks **on** the things it names. Concretely: predicted
> masks must beat a **null baseline** — the same masks randomly translated — on
> classes where we have trusted labels. If it cannot find buildings that OSM
> proves are there, it cannot be trusted on `track` and `woody`, which have no
> yardstick at all.

That last point is why this probe is worth running despite a bad prior: it is
**self-falsifying**, and it costs no labelling and no training. Probe 1 lacked
that property and had to be rejected by eye.

## How to run

From `src/` with the `uavloc` env. Long runs:

```
papermill pocs/class-label-sources/yoloe_satellite.ipynb \
          pocs/class-label-sources/runs/yoloe-$(date +%F).ipynb -p n_locations 9
```

In [ ]:
# papermill parameters
n_locations = 2       # committed results used 9
n_buildings = 3       # per location, for the small-crop test; committed used 8
conf        = 0.05
tile        = 1024
overlap     = 128
variant     = "yoloe-11l-seg.pt"
device      = "cpu"   # measured: CPU beats MPS on this Intel machine
extents_m   = [25, 50, 100, 200]

## Setup

In [ ]:
import json, sys, time, warnings
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image, ImageDraw
from scipy import ndimage
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

SRC  = Path.cwd()
if SRC.name != "src":                       # papermill may start elsewhere
    SRC = next(p for p in [SRC, *SRC.parents] if (p / "uavloc").is_dir())
ROOT = SRC.parent / "data" / "derived" / "ua-village-multidate"
HERE = SRC / "pocs" / "class-label-sources"
FIGS = HERE / "figures"; FIGS.mkdir(exist_ok=True)

# shared helper, imported because three PoCs use it -- true Web Mercator geometry
sys.path.insert(0, str(SRC / "pocs" / "map-derived-labels"))
from tiles import TileGrid

print("src :", SRC)
print("data:", ROOT, "-", len(json.loads((ROOT / "index.json").read_text())["built"]), "locations")

## The data

One location is a ~1284 x 988 m box of Esri imagery at Web Mercator **z18
(0.386 m/px)**, with an OSM class mask (`label.png`) rasterised through the same
tile grid. `(z, x, y)` *is* a box, so pixel to lat/lon is closed form — that is
what makes any model run over this tile coordinate-addressable, and therefore a
legitimate label source.

In [ ]:
def load_location(name):
    meta  = json.loads((ROOT / name / "meta.json").read_text())
    image = Image.open(ROOT / name / "current.jpg").convert("RGB")
    label = np.array(Image.open(ROOT / name / "label.png"))
    return image, label, meta

# the 9 locations probe 1 used, so the two probes are comparable
names = list(json.loads((HERE / "results-field.json").read_text()))
names = names[:n_locations]

image, label, meta = load_location(names[0])
print(names[0])
print(f"  {image.size[0]}x{image.size[1]} px = {image.size[0]*meta['mpp']:.0f} x "
      f"{image.size[1]*meta['mpp']:.0f} m  at {meta['mpp']:.3f} m/px")
print(f"  OSM drew: {meta['osm']['drawn']}")
image.resize((image.size[0]//4, image.size[1]//4))

## First attempt: just name the classes

The obvious thing, and what the whole idea rests on.

In [ ]:
from ultralytics import YOLOE

model = YOLOE(variant)
patch = np.array(image.crop((1200, 1000, 2224, 2024)))   # 391 m of village centre

def try_prompts(prompt_names, patch=patch, conf=conf):
    model.set_classes(prompt_names, model.get_text_pe(prompt_names))
    r = model.predict(patch, imgsz=1024, conf=conf, device=device,
                      verbose=False, retina_masks=True)[0]
    n = 0 if r.masks is None else len(r.masks.xy)
    got = [] if r.boxes is None else [prompt_names[int(c)].split()[0] for c in r.boxes.cls]
    cf  = [] if r.boxes is None else [float(c) for c in r.boxes.conf]
    return n, dict(Counter(got)), (round(max(cf), 2) if cf else 0.0)

print(try_prompts(["building", "dirt road", "trees", "river"]))

**Zero masks.** Not a single detection at `conf=0.05` over a village full of
buildings. Before concluding anything, vary the phrasing — CLIP text encoders
are sensitive to it.

In [ ]:
PROMPTS = {
    "plain":     ["building", "dirt road", "trees", "river"],
    "aerial":    ["aerial view of a building", "dirt road seen from above",
                  "group of trees from above", "river seen from above"],
    "satellite": ["building rooftop in satellite image", "unpaved track in satellite image",
                  "forest in satellite image", "water body in satellite image"],
}
for tag, p in PROMPTS.items():
    n, got, mx = try_prompts(p)
    print(f"{tag:10s} -> {n:2d} masks  {got}  max_conf={mx}")

Phrasing dominates everything: bare nouns return nothing, the
`"... in satellite image"` suffix is what makes the encoder fire. **Everything
below uses `satellite` — the most favourable phrasing found**, so the probe is
not failing for want of prompt engineering.

## The most diagnostic single result

Prompt-free mode lets the model name what it sees with no prompt at all. This is
the cheapest possible read on whether it understands the domain.

In [ ]:
pf = YOLOE("yoloe-11l-seg-pf.pt")
r  = pf.predict(patch, imgsz=1024, conf=conf, device=device,
                verbose=False, retina_masks=True)[0]
labels = [] if r.boxes is None else [pf.names[int(c)] for c in r.boxes.cls]
print(f"{0 if r.masks is None else len(r.masks.xy)} masks over a Ukrainian village:")
print(Counter(labels).most_common(12))

Over farmland and farmhouses it returns **fish, scenery, aircraft carrier,
icicle, shipwreck**. Not one correct label. The model is not mislabelling our
classes — it does not recognise the scene as land seen from above, and parses it
as something aquatic.

That is the explanation for everything that follows: **a prompt retrieves a
concept the encoder already holds; it cannot teach one.** These vocabularies
come from ground-level photographs (LVIS, Objects365, GoldG).

## The control

`building` and `water` come from OSM and are measured good, so they are the
yardstick. The mask is shifted by the per-location offset the dataset build
recorded — labels sit a median 2.84 m off because OSM was traced against a
different basemap, and charging that to the detector would be unfair.

In [ ]:
OSM_ID  = {"building": 1, "track": 2, "water": 3, "woody": 4}
CLASSES = ["building", "track", "woody", "water"]      # YOLOE class order

def control_masks(label, meta):
    off = meta.get("label_offset") or {}
    dx, dy = int(off.get("dx_px", 0)), int(off.get("dy_px", 0))
    shifted = np.roll(np.roll(label, -dy, axis=0), -dx, axis=1)
    return {k: (shifted == v) for k, v in OSM_ID.items()}

def burn(polys, shape):
    canvas = Image.new("L", (shape[1], shape[0]), 0)
    draw = ImageDraw.Draw(canvas)
    for p in polys:
        pts = [tuple(map(float, q)) for q in p]
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)
    return np.array(canvas, dtype=bool)

ctrl = control_masks(label, meta)
for k, v in ctrl.items():
    print(f"{k:9s} {v.mean()*100:5.2f}% of tile")

## Tiled inference

Not the whole image — 1024 px tiles at native 0.386 m/px, so **391 m of ground
per call**, 128 px overlap. A polygon counts only if its centroid lands in the
tile core, otherwise every seam yields two copies of the same feature.

In [ ]:
def predict_multiclass(model, image, tile=tile, overlap=overlap, conf=conf):
    W, H, step = *image.size, tile - overlap
    found = []
    for top in range(0, max(1, H - overlap), step):
        for left in range(0, max(1, W - overlap), step):
            right, bottom = min(left + tile, W), min(top + tile, H)
            crop = np.array(image.crop((left, top, right, bottom)))
            if min(crop.shape[:2]) < 32:
                continue
            r = model.predict(crop, imgsz=tile, conf=conf, device=device,
                              verbose=False, retina_masks=True)[0]
            if r.masks is None or r.boxes is None:
                continue
            cx0 = left + overlap/2 if left > 0 else -np.inf
            cx1 = right - overlap/2 if right < W else np.inf
            cy0 = top + overlap/2 if top > 0 else -np.inf
            cy1 = bottom - overlap/2 if bottom < H else np.inf
            for poly, cls, cf in zip(r.masks.xy, r.boxes.cls, r.boxes.conf):
                if len(poly) < 3:
                    continue
                s = poly + np.array([left, top])
                x, y = s.mean(axis=0)
                if cx0 <= x <= cx1 and cy0 <= y <= cy1:
                    found.append((int(cls), s, float(cf)))
    return found

model.set_classes(PROMPTS["satellite"], model.get_text_pe(PROMPTS["satellite"]))
t = time.time()
found = predict_multiclass(model, image)
print(f"{len(found)} detections in {time.time()-t:.0f}s -> "
      f"{dict(Counter(CLASSES[c] for c,_,_ in found))}")

## The trap, and the null baseline

Instance recall here is deliberately generous — an OSM blob counts as found if
*any* predicted pixel touches it. Run it straight and the number looks usable.
Then look at how much of the tile the prediction covers.

In [ ]:
def instance_recall(control, predicted, min_px=20):
    lab, n = ndimage.label(control)
    if n == 0:
        return None, 0
    sizes = ndimage.sum(control, lab, range(1, n + 1))
    keep = [i + 1 for i, s in enumerate(sizes) if s >= min_px]
    if not keep:
        return None, 0
    return sum(1 for i in keep if predicted[lab == i].any()) / len(keep), len(keep)

shape = tuple(meta["shape"])
pred_b = burn([p for c, p, _ in found if c == 0], shape)
rec, tot = instance_recall(ctrl["building"], pred_b)
print(f"building instance recall : {rec:.2f} of {tot} OSM buildings")
print(f"...but the prediction covers {pred_b.mean()*100:.0f}% of the tile, "
      f"where buildings are {ctrl['building'].mean()*100:.1f}%")

A mask covering half the image touches most things in it. So every score gets a
**null**: the same prediction randomly circular-shifted, which preserves blob
count, shape and total coverage and destroys only *where* the blobs are.

Anything that does not beat its null is an artifact of area painted. This is the
guard probe 1 did not have — see `bug-014`, `bug-016`.

In [ ]:
def null_recall(control, predicted, trials=20, seed=0):
    rng = np.random.default_rng(seed)
    H, W = predicted.shape
    out = []
    for _ in range(trials):
        rolled = np.roll(np.roll(predicted, rng.integers(0, H), 0), rng.integers(0, W), 1)
        v, _ = instance_recall(control, rolled)
        if v is not None:
            out.append(v)
    return float(np.mean(out)) if out else None

print(f"real recall {rec:.2f}   vs   null {null_recall(ctrl['building'], pred_b):.2f}")

## Over every location

In [ ]:
def score(found, control, shape):
    out = {}
    for i, name in enumerate(CLASSES):
        polys = [p for c, p, _ in found if c == i]
        confs = [f for c, _, f in found if c == i]
        pred, truth = burn(polys, shape), control[name]
        r, tot = instance_recall(truth, pred)
        union = (pred | truth).sum()
        out[name] = {"dets": len(polys), "cover": float(pred.mean()),
                     "osm": float(truth.mean()), "n": tot, "recall": r,
                     "null": null_recall(truth, pred) if r is not None else None,
                     "iou": float((pred & truth).sum()/union) if union else None,
                     "max_conf": round(max(confs), 3) if confs else 0.0}
    return out

results = {}
for i, nm in enumerate(names, 1):
    im, lb, mt = load_location(nm)
    f = predict_multiclass(model, im)
    results[nm] = score(f, control_masks(lb, mt), tuple(mt["shape"]))
    b = results[nm]["building"]
    print(f"[{i}/{len(names)}] {nm.split('-')[0]}: recall {b['recall']:.2f} vs null "
          f"{b['null']:.2f}, cover {b['cover']*100:.0f}%, IoU {b['iou']:.3f}")

In [ ]:
print(f"{'class':9s} {'dets':>5s} {'cover%':>7s} {'osm%':>6s} {'recall':>7s} {'null':>6s} {'IoU':>6s}")
for c in CLASSES:
    g = lambda k: [results[l][c][k] for l in results if results[l][c][k] is not None]
    r, n = g("recall"), g("null")
    print(f"{c:9s} {sum(results[l][c]['dets'] for l in results):5d} "
          f"{np.mean(g('cover'))*100:7.1f} {np.mean(g('osm'))*100:6.2f} "
          f"{(np.mean(r) if r else 0):7.2f} {(np.mean(n) if n else 0):6.2f} "
          f"{np.mean(g('iou')):6.3f}")

With all 9 locations this reads: `building` 0.77 vs null 0.61,
`track` **0.09 vs 0.10**, `woody` **0.27 vs 0.26**, `water` **0.25 vs 0.33**.

Three of four classes at or *below* chance. `building` clears its null only by
claiming 55.7% of the ground when buildings occupy 4.09% — IoU 0.046. `track`,
the class the flight ground most needs, produced **10 detections across all nine
locations**.

## Can it draw a small box, and where do the masks land?

Looking at the overview figure, the masks are roughly tile-sized. That suggests a
different failure from "wrong concept": maybe the model simply emits one blanket
box per frame, and a smaller crop containing a single building would get a usable
one. Testable, and a different claim — so test it.

Crops centred on one known OSM building, at four ground extents, all resized to
640 so the only variable is how much ground is in frame.

**Record every mask, not the largest.** Summarising a detector by its extremum
produced a false claim once already (`bug-019`).

In [ ]:
def building_centres(control, want, min_px, rng):
    lab, n = ndimage.label(control)
    if n == 0:
        return []
    sizes = ndimage.sum(control, lab, range(1, n + 1))
    keep = [i + 1 for i, s in enumerate(sizes) if s >= min_px]
    out = []
    for idx in rng.choice(keep, size=min(want, len(keep)), replace=False):
        ys, xs = np.where(lab == idx)
        out.append((int(xs.mean()), int(ys.mean()), lab == idx))
    return out

IMGSZ, rng, rows = 640, np.random.default_rng(0), []
for nm in names:
    im, lb, mt = load_location(nm)
    c, mpp = control_masks(lb, mt)["building"], mt["mpp"]
    H, W = mt["shape"]
    for cx, cy, blob in building_centres(c, n_buildings, 60, rng):
        for e in extents_m:
            half = int(round(e / mpp / 2))
            x0, y0 = max(0, cx-half), max(0, cy-half)
            x1, y1 = min(W, cx+half), min(H, cy+half)
            if x1-x0 < 16 or y1-y0 < 16:
                continue
            crop = im.crop((x0, y0, x1, y1)).resize((IMGSZ, IMGSZ), Image.LANCZOS)
            tgt = np.array(Image.fromarray(blob[y0:y1, x0:x1].astype("uint8")*255)
                           .resize((IMGSZ, IMGSZ), Image.NEAREST)) > 127
            r = model.predict(np.array(crop), imgsz=IMGSZ, conf=conf, device=device,
                              verbose=False, retina_masks=True)[0]
            if r.masks is None:
                continue
            for poly in r.masks.xy:
                m = burn([poly], (IMGSZ, IMGSZ))
                inter, union = (m & tgt).sum(), (m | tgt).sum()
                rows.append({"extent": e, "frac": float(m.mean()),
                             "iou": float(inter/union) if union else 0.0,
                             "bldg": float(tgt.mean())})
    print("done", nm.split('-')[0], flush=True)
print(f"\n{len(rows)} masks recorded")

In [ ]:
a = np.array([[r["extent"], r["frac"], r["iou"], r["bldg"]] for r in rows])
print(f"{'extent':>7s} {'masks':>6s} {'median frac':>12s} {'<30% frame':>11s} {'best IoU':>9s}")
for e in extents_m:
    s = a[a[:,0] == e]
    if len(s):
        print(f"{e:7.0f} {len(s):6d} {np.median(s[:,1]):12.3f} "
              f"{100*np.mean(s[:,1]<0.3):10.1f}% {s[:,2].max():9.3f}")

small = a[a[:,1] < 0.3]
print(f"\nmasks under 30% of frame: {len(small)}/{len(a)} ({100*len(small)/len(a):.1f}%)")
if len(small):
    print(f"  their IoU: median {np.median(small[:,2]):.3f}")
    print(f"  small AND well-placed (IoU>0.3): {int((small[:,2] > 0.3).sum())}")

With the full run: 27.3% of 417 masks are under 30% of frame, so **the model
*can* draw a small box** — the first version of this probe claimed otherwise from
a largest-mask-only figure and was wrong (`bug-019`, retracted).

What survives is narrower and worse: the small masks have **median IoU 0.000**,
and only **7 of 417** are both small and well-placed. They trace roads and field
edges while the target building sits untouched. Mean best-IoU per crop falls
monotonically with context — 0.193 / 0.067 / 0.046 / 0.003 — so tight-and-wrong
at one end, blanket at the other. **No extent turns this into a label source**,
which is what closes the scale hypothesis.

## Look at it

Full resolution, every mask, before the verdict.

In [ ]:
im, lb, mt = load_location(names[0])
c, mpp = control_masks(lb, mt)["building"], mt["mpp"]
H, W = mt["shape"]
picks = building_centres(c, 3, 200, np.random.default_rng(3))

fig, axes = plt.subplots(len(extents_m), len(picks),
                         figsize=(3.8*len(picks), 3.8*len(extents_m)), squeeze=False)
for row, e in enumerate(extents_m):
    for col, (cx, cy, blob) in enumerate(picks):
        half = int(round(e/mpp/2))
        x0, y0 = max(0, cx-half), max(0, cy-half)
        x1, y1 = min(W, cx+half), min(H, cy+half)
        crop = im.crop((x0, y0, x1, y1)).resize((IMGSZ, IMGSZ), Image.LANCZOS)
        tgt = np.array(Image.fromarray(blob[y0:y1, x0:x1].astype("uint8")*255)
                       .resize((IMGSZ, IMGSZ), Image.NEAREST)) > 127
        r = model.predict(np.array(crop), imgsz=IMGSZ, conf=conf, device=device,
                          verbose=False, retina_masks=True)[0]
        ax = axes[row][col]
        ax.imshow(crop); ax.contour(tgt, [0.5], colors="#3cb44b", linewidths=2)
        best, n_small = 0.0, 0
        if r.masks is not None:
            for poly in r.masks.xy:
                m = burn([poly], (IMGSZ, IMGSZ)); f = m.mean()
                n_small += f <= 0.5
                u = (m | tgt).sum()
                best = max(best, (m & tgt).sum()/u if u else 0)
                ax.contour(m, [0.5], colors="#e6194b" if f > 0.5 else "#ff8c00",
                           linewidths=2 if f > 0.5 else 1.4)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{n_small} small · best IoU {best:.2f}", fontsize=9)
        if col == 0:
            ax.set_ylabel(f"{e:.0f} m", fontsize=12)
fig.suptitle("green = OSM building · red = mask >50% of frame · orange = smaller", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## Verdict — `failed`

Open-vocabulary prompting cannot label these classes. Three of four sit at or
below a position-destroying null; the fourth clears it only by painting half the
tile. The cause is not tuning and not object scale: **the vocabulary was learned
from ground-level photographs and nadir imagery is outside it**, which the
prompt-free output (fish, aircraft carrier, shipwreck) shows directly. A prompt
retrieves a concept the encoder holds; it cannot teach one.

### What is *not* established

- **Geospatial foundation models were not tested.** This verdict covers
  YOLOE/YOLO-World checkpoints trained on natural images, a different prior.
- **Visual prompting was only asked for `building`.** It collapsed (recall 0.26
  vs null 0.25, detections swinging 0 to 403 per location), but `track` and
  `woody` were never prompted visually.
- **`conf=0.05` is a floor, not tuned.** Raising it only removes detections, and
  the failure is that detections are everywhere — but it was not swept.

Copied into `notes.md` as probe 4. Full-run artifacts:
`results-yoloe-text-satellite.json`, `results-yoloe-allmasks.json`,
`figures/yoloe-overview.png`, `figures/yoloe-scores.png`,
`figures/yoloe-smallcrop.png`.